1. Google Drive csatolása

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


2. Importok, eszköz, konfiguráció, pathok

In [ ]:
import os
import random
from dataclasses import dataclass
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split

import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Adat gyökér (ugyanaz, mint eddig)
root_path = "/content/drive/MyDrive/Brain MRI/brisc2025/segmentation_task/train"

# IDE mentünk mindent az Attention U-Net FNO-hoz
output_root = "/content/drive/MyDrive/Brain MRI/AttentionUnet_FNO"

@dataclass
class Config:
    # seedek
    seed_global: int = 42
    seed_split: int = 123

    # tréning
    batch_size: int = 4
    num_epochs: int = 100
    lr: float = 1e-3
    weight_decay: float = 1e-5
    num_workers: int = 2
    val_ratio: float = 0.2
    evals_per_epoch: int = 4

    # warmup + scheduler
    use_warmup: bool = True
    warmup_epochs: int = 3
    warmup_start_lr: float = 1e-5
    scheduler_type: str = "cosine"  # "cosine" vagy "none"
    eta_min: float = 1e-5           # cosine min lr

    # képméret, osztályok
    image_height: int = 256
    image_width: int = 256
    num_classes: int = 2

    # loss
    loss_bce_weight: float = 0.5
    loss_dice_weight: float = 0.5
    loss_smooth: float = 1.0

    # pathok
    model_dir: str = os.path.join(output_root, "models")
    plots_dir: str = os.path.join(output_root, "plots")
    preds_dir: str = os.path.join(output_root, "predictions")
    image_dir: str = os.path.join(root_path, "images")
    mask_dir: str = os.path.join(root_path, "masks")

cfg = Config()

print("Images dir:", cfg.image_dir, "exists:", os.path.isdir(cfg.image_dir))
print("Masks dir:", cfg.mask_dir, "exists:", os.path.isdir(cfg.mask_dir))

os.makedirs(cfg.model_dir, exist_ok=True)
os.makedirs(cfg.plots_dir, exist_ok=True)
os.makedirs(cfg.preds_dir, exist_ok=True)


Device: cuda
Images dir: /content/drive/MyDrive/Brain MRI/brisc2025/segmentation_task/train/images exists: True
Masks dir: /content/drive/MyDrive/Brain MRI/brisc2025/segmentation_task/train/masks exists: True


3. Seed fixálás

In [ ]:
def set_seed(seed: int):
    print(f"*** Setting global seed to {seed} ***")
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(cfg.seed_global)


*** Setting global seed to 42 ***


4. Dataset

In [ ]:
class SegmentationDataset(Dataset):
    def __init__(self, image_dir: str, mask_dir: str):
        self.image_dir = image_dir
        self.mask_dir = mask_dir

        self.image_files = sorted([
            f for f in os.listdir(image_dir)
            if f.lower().endswith((".png", ".jpg", ".jpeg", ".tif"))
        ])
        self.mask_files = sorted([
            f for f in os.listdir(mask_dir)
            if f.lower().endswith((".png", ".jpg", ".jpeg", ".tif"))
        ])

        if len(self.image_files) == 0:
            raise ValueError(f"Nincs kép a(z) {image_dir} mappában!")
        if len(self.mask_files) == 0:
            raise ValueError(f"Nincs maszk a(z) {mask_dir} mappában!")
        assert len(self.image_files) == len(self.mask_files), \
            "A képek és maszkok száma nem egyezik!"

        print(f"Találtam {len(self.image_files)} képet és maszkot.")

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx: int):
        img_path = os.path.join(self.image_dir, self.image_files[idx])
        mask_path = os.path.join(self.mask_dir, self.mask_files[idx])

        image = Image.open(img_path).convert("L")
        mask = Image.open(mask_path).convert("L")

        image = image.resize((cfg.image_width, cfg.image_height))
        mask = mask.resize((cfg.image_width, cfg.image_height), Image.NEAREST)

        image = np.array(image, dtype=np.float32) / 255.0
        mask = np.array(mask, dtype=np.int64)
        mask = (mask > 127).astype(np.int64)

        image = np.expand_dims(image, axis=0)

        return torch.from_numpy(image), torch.from_numpy(mask)


5. Train/val split + DataLoader

In [ ]:
full_dataset = SegmentationDataset(cfg.image_dir, cfg.mask_dir)
dataset_size = len(full_dataset)
val_size = int(dataset_size * cfg.val_ratio)
train_size = dataset_size - val_size

print("Összes minta:", dataset_size)
print("Train:", train_size, "Val:", val_size)

set_seed(cfg.seed_split)
generator = torch.Generator().manual_seed(cfg.seed_split)

from torch.utils.data import random_split, DataLoader

train_dataset, val_dataset = random_split(
    full_dataset,
    [train_size, val_size],
    generator=generator
)

train_loader = DataLoader(
    train_dataset,
    batch_size=cfg.batch_size,
    shuffle=True,
    num_workers=cfg.num_workers,
    pin_memory=True
)
val_loader = DataLoader(
    val_dataset,
    batch_size=cfg.batch_size,
    shuffle=False,
    num_workers=cfg.num_workers,
    pin_memory=True
)

images, masks = next(iter(train_loader))
print("Batch images:", images.shape)
print("Batch masks:", masks.shape)


Találtam 3933 képet és maszkot.
Összes minta: 3933
Train: 3147 Val: 786
*** Setting global seed to 123 ***
Batch images: torch.Size([4, 1, 256, 256])
Batch masks: torch.Size([4, 256, 256])


6. Fourier + Attention U‑Net FNO modell

In [ ]:
@torch.jit.script
def compl_mul2d(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    return torch.einsum("bixy, ioxy->boxy", a, b)


class SpectralConv2d(nn.Module):
    def __init__(self, in_channels, out_channels, modes1, modes2):
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.modes1 = modes1
        self.modes2 = modes2
        self.scale = 1 / (in_channels * out_channels)

        self.weights1 = nn.Parameter(
            self.scale * torch.rand(in_channels, out_channels, self.modes1, self.modes2, dtype=torch.cfloat)
        )
        self.weights2 = nn.Parameter(
            self.scale * torch.rand(in_channels, out_channels, self.modes1, self.modes2, dtype=torch.cfloat)
        )

    def forward(self, x):
        batchsize = x.shape[0]
        x_ft = torch.fft.rfftn(x, dim=[2, 3])

        out_ft = torch.zeros(
            batchsize,
            self.out_channels,
            x.size(-2),
            x.size(-1) // 2 + 1,
            device=x.device,
            dtype=torch.cfloat
        )

        out_ft[:, :, :self.modes1, :self.modes2] = \
            compl_mul2d(x_ft[:, :, :self.modes1, :self.modes2], self.weights1)
        out_ft[:, :, -self.modes1:, :self.modes2] = \
            compl_mul2d(x_ft[:, :, -self.modes1:, :self.modes2], self.weights2)

        x = torch.fft.irfftn(out_ft, s=(x.size(-2), x.size(-1)), dim=[2, 3])
        return x


class FourierLayer(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, padding=1, modes=4):
        super().__init__()
        self.conv = nn.Conv2d(
            in_channels=in_channels,
            out_channels=out_channels,
            kernel_size=kernel_size,
            stride=stride,
            padding=padding,
            bias=True
        )
        self.conv_fno = SpectralConv2d(in_channels, out_channels, modes, modes)

    def forward(self, x):
        x1 = self.conv(x)
        x2 = self.conv_fno(x)
        return x1 + x2


class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch, modes=4):
        super().__init__()
        self.net = nn.Sequential(
            FourierLayer(in_ch, out_ch, kernel_size=3, stride=1, padding=1, modes=modes),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            FourierLayer(out_ch, out_ch, kernel_size=3, stride=1, padding=1, modes=modes),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class AttentionBlock(nn.Module):
    def __init__(self, g_ch, x_ch, inter_ch):
        super().__init__()
        self.W_g = nn.Sequential(
            nn.Conv2d(g_ch, inter_ch, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm2d(inter_ch),
        )
        self.W_x = nn.Sequential(
            nn.Conv2d(x_ch, inter_ch, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm2d(inter_ch),
        )
        self.psi = nn.Sequential(
            nn.Conv2d(inter_ch, 1, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm2d(1),
            nn.Sigmoid(),
        )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, g, x):
        g1 = self.W_g(g)
        x1 = self.W_x(x)
        psi = self.relu(g1 + x1)
        psi = self.psi(psi)
        return x * psi


class AttentionUNetFNO(nn.Module):
    def __init__(self, in_channels=1, out_channels=2, modes=4):
        super().__init__()
        # Encoder
        self.down1 = DoubleConv(in_channels, 64, modes=modes)
        self.pool1 = nn.MaxPool2d(2)

        self.down2 = DoubleConv(64, 128, modes=modes)
        self.pool2 = nn.MaxPool2d(2)

        # Bridge
        self.bridge = DoubleConv(128, 256, modes=modes)

        # Attention blokkok
        self.att2 = AttentionBlock(g_ch=128, x_ch=128, inter_ch=64)
        self.att1 = AttentionBlock(g_ch=64, x_ch=64, inter_ch=32)

        # Decoder
        self.up2 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.conv2 = DoubleConv(256, 128, modes=modes)

        self.up1 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.conv1 = DoubleConv(128, 64, modes=modes)

        self.out_conv = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x):
        d1 = self.down1(x)
        p1 = self.pool1(d1)

        d2 = self.down2(p1)
        p2 = self.pool2(d2)

        b = self.bridge(p2)

        u2 = self.up2(b)
        d2_att = self.att2(g=u2, x=d2)
        u2 = torch.cat([u2, d2_att], dim=1)
        u2 = self.conv2(u2)

        u1 = self.up1(u2)
        d1_att = self.att1(g=u1, x=d1)
        u1 = torch.cat([u1, d1_att], dim=1)
        u1 = self.conv1(u1)

        out = self.out_conv(u1)
        return out


# gyors teszt
model = AttentionUNetFNO(in_channels=1, out_channels=cfg.num_classes, modes=4).to(device)
x_test = torch.randn(4, 1, cfg.image_height, cfg.image_width).to(device)
y_test = model(x_test)
print("Model output shape:", y_test.shape)
print("Total params:", sum(p.numel() for p in model.parameters()))


Model output shape: torch.Size([4, 2, 256, 256])
Total params: 7916840


7. Loss (Combined BCE + Dice)

In [ ]:
class CombinedBCEDiceLoss(nn.Module):
    def __init__(self, bce_weight=0.5, dice_weight=0.5, smooth=1.0):
        super().__init__()
        self.bce_weight = bce_weight
        self.dice_weight = dice_weight
        self.smooth = smooth
        self.ce_loss = nn.CrossEntropyLoss()

    def dice_loss(self, logits, targets, num_classes=2):
        probs = torch.softmax(logits, dim=1)
        targets_one_hot = torch.nn.functional.one_hot(targets, num_classes=num_classes)
        targets_one_hot = targets_one_hot.permute(0, 3, 1, 2).float()

        dice_scores = []
        for cls in range(num_classes):
            pred_cls = probs[:, cls, :, :]
            target_cls = targets_one_hot[:, cls, :, :]
            intersection = (pred_cls * target_cls).sum()
            union = pred_cls.sum() + target_cls.sum()
            dice = (2.0 * intersection + self.smooth) / (union + self.smooth)
            dice_scores.append(dice)
        mean_dice = sum(dice_scores) / len(dice_scores)
        return 1.0 - mean_dice

    def forward(self, logits, targets):
        ce_loss = self.ce_loss(logits, targets)
        dice_loss = self.dice_loss(logits, targets, num_classes=logits.size(1))
        return self.bce_weight * ce_loss + self.dice_weight * dice_loss


8. Metrikák (Dice, IoU)

In [ ]:
def dice_score(logits, targets, num_classes=2):
    preds = torch.argmax(logits, dim=1)
    batch_size = preds.size(0)
    sample_dice_scores = []
    for i in range(batch_size):
        dice_scores = []
        for cls in range(num_classes):
            pred_mask = (preds[i] == cls).float()
            target_mask = (targets[i] == cls).float()
            intersection = (pred_mask * target_mask).sum()
            dice = (2.0 * intersection) / (pred_mask.sum() + target_mask.sum() + 1e-8)
            dice_scores.append(dice.item())
        sample_dice_scores.append(sum(dice_scores) / len(dice_scores))
    return sample_dice_scores


def compute_iou(logits, targets, num_classes=2):
    preds = torch.argmax(logits, dim=1)
    batch_size = preds.size(0)
    sample_iou_scores = []
    for i in range(batch_size):
        ious = []
        for cls in range(num_classes):
            pred_inds = (preds[i] == cls)
            target_inds = (targets[i] == cls)
            intersection = (pred_inds & target_inds).sum().item()
            union = (pred_inds | target_inds).sum().item()
            if union == 0:
                continue
            ious.append(intersection / union)
        if len(ious) == 0:
            sample_iou_scores.append(0.0)
        else:
            sample_iou_scores.append(sum(ious) / len(ious))
    return sample_iou_scores


9. Warmup LR + validáció + tréning loop

In [ ]:
def get_lr_for_epoch(epoch, base_lr, cfg: Config):
    if cfg.use_warmup and epoch < cfg.warmup_epochs:
        alpha = epoch / max(1, cfg.warmup_epochs)
        return cfg.warmup_start_lr + alpha * (base_lr - cfg.warmup_start_lr)
    return base_lr


@torch.no_grad()
def validate(model, loader, criterion):
    model.eval()
    running_loss = 0.0
    total_samples = 0
    all_dice = []
    all_iou = []

    for images, masks in loader:
        images = images.to(device)
        masks = masks.to(device)
        outputs = model(images)
        loss = criterion(outputs, masks)

        running_loss += loss.item() * images.size(0)
        total_samples += images.size(0)

        dice_scores = dice_score(outputs, masks, num_classes=cfg.num_classes)
        iou_scores = compute_iou(outputs, masks, num_classes=cfg.num_classes)

        all_dice.extend(dice_scores)
        all_iou.extend(iou_scores)

    val_loss = running_loss / total_samples
    val_dice = sum(all_dice) / len(all_dice)
    val_iou = sum(all_iou) / len(all_iou)
    return val_loss, val_dice, val_iou


def train_model_with_logging(run_name: str, seed: int):
    set_seed(seed)

    model = AttentionUNetFNO(
        in_channels=1,
        out_channels=cfg.num_classes,
        modes=4
    ).to(device)

    print("Using Attention U-Net FNO model")
    print("Total params:", sum(p.numel() for p in model.parameters()))

    criterion = CombinedBCEDiceLoss(
        bce_weight=cfg.loss_bce_weight,
        dice_weight=cfg.loss_dice_weight,
        smooth=cfg.loss_smooth
    )
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

    batches_per_epoch = len(train_loader)

    if cfg.scheduler_type == "cosine":
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=cfg.num_epochs,
            eta_min=cfg.eta_min
        )
        print(f"Using CosineAnnealingLR: T_max={cfg.num_epochs}, eta_min={cfg.eta_min}")
    else:
        scheduler = None
        print("No scheduler, constant lr (with optional warmup)")

    print(f"Evaluations per epoch: {cfg.evals_per_epoch}")
    eval_interval = max(1, batches_per_epoch // cfg.evals_per_epoch)
    print(f"Evaluation interval: every {eval_interval} batches")

    train_history = []
    val_history = []
    best_val_dice = 0.0

    for epoch in range(1, cfg.num_epochs + 1):
        model.train()
        running_loss = 0.0
        total_samples = 0
        all_dice_scores = []
        batch_count = 0

        epoch_lr = get_lr_for_epoch(epoch - 1, cfg.lr, cfg)
        for param_group in optimizer.param_groups:
            param_group["lr"] = epoch_lr

        for images, masks in train_loader:
            images = images.to(device)
            masks = masks.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, masks)
            loss.backward()
            optimizer.step()

            dice_scores = dice_score(outputs.detach(), masks, num_classes=cfg.num_classes)
            all_dice_scores.extend(dice_scores)

            running_loss += loss.item() * images.size(0)
            total_samples += images.size(0)
            batch_count += 1

            if batch_count % eval_interval == 0 or batch_count == batches_per_epoch:
                train_loss = running_loss / total_samples
                train_dice = sum(all_dice_scores) / len(all_dice_scores)
                val_loss, val_dice, val_iou = validate(model, val_loader, criterion)

                current_lr = optimizer.param_groups[0]["lr"]
                step_in_epoch = batch_count / batches_per_epoch
                print(
                    f"Epoch {epoch:02d} step {step_in_epoch:.2f} | "
                    f"lr={current_lr:.6f} | "
                    f"train_loss={train_loss:.4f} | train_dice={train_dice:.4f} | "
                    f"val_loss={val_loss:.4f} | val_dice={val_dice:.4f} | val_iou={val_iou:.4f}"
                )

                train_history.append((train_loss, train_dice))
                val_history.append((val_loss, val_dice, val_iou))

                if val_dice > best_val_dice:
                    best_val_dice = val_dice
                    save_path = os.path.join(cfg.model_dir, f"{run_name}_best_model.pt")
                    torch.save({
                        "epoch": epoch,
                        "model_state_dict": model.state_dict(),
                        "optimizer_state_dict": optimizer.state_dict(),
                        "val_dice": val_dice,
                        "val_iou": val_iou,
                        "seed": seed,
                    }, save_path)
                    print(f"Új legjobb modell mentve: epoch {epoch}, Dice={val_dice:.4f}, IoU={val_iou:.4f}")

        if scheduler is not None:
            scheduler.step()

    return {
        "run_name": run_name,
        "seed": seed,
        "train_history": train_history,
        "val_history": val_history,
        "best_val_dice": best_val_dice
    }


10. Tréning, görbék, predikció

In [ ]:
# 10.1 Tréning
result_att_fno = train_model_with_logging(run_name="run1_att_fno", seed=cfg.seed_global)
print("Best val Dice (Attention U-Net FNO):", result_att_fno["best_val_dice"])


*** Setting global seed to 42 ***
Using Attention U-Net FNO model
Total params: 7916840
Using CosineAnnealingLR: T_max=100, eta_min=1e-05
Evaluations per epoch: 4
Evaluation interval: every 196 batches
Epoch 01 step 0.25 | lr=0.000010 | train_loss=0.4816 | train_dice=0.6042 | val_loss=0.4106 | val_dice=0.7010 | val_iou=0.6455
Új legjobb modell mentve: epoch 1, Dice=0.7010, IoU=0.6455
Epoch 01 step 0.50 | lr=0.000010 | train_loss=0.3223 | train_dice=0.6692 | val_loss=0.1348 | val_dice=0.7771 | val_iou=0.7229
Új legjobb modell mentve: epoch 1, Dice=0.7771, IoU=0.7229
Epoch 01 step 0.75 | lr=0.000010 | train_loss=0.2597 | train_dice=0.7054 | val_loss=0.1227 | val_dice=0.7886 | val_iou=0.7387
Új legjobb modell mentve: epoch 1, Dice=0.7886, IoU=0.7387
Epoch 01 step 1.00 | lr=0.000010 | train_loss=0.2239 | train_dice=0.7286 | val_loss=0.1164 | val_dice=0.8073 | val_iou=0.7555
Új legjobb modell mentve: epoch 1, Dice=0.8073, IoU=0.7555
Epoch 01 step 1.00 | lr=0.000010 | train_loss=0.2235 | tra

In [ ]:
# 10.2 Loss / Dice / IoU görbék mentése

train_losses = np.array(result_att_fno["train_history"])[:, 0]
train_dices = np.array(result_att_fno["train_history"])[:, 1]
val_losses = np.array(result_att_fno["val_history"])[:, 0]
val_dices = np.array(result_att_fno["val_history"])[:, 1]
val_ious = np.array(result_att_fno["val_history"])[:, 2]

epochs = np.arange(1, len(train_losses) + 1)

plt.figure(figsize=(6, 4))
plt.plot(epochs, train_losses, label="Train loss")
plt.plot(epochs, val_losses, label="Val loss")
plt.legend()
plt.grid(True)
plt.xlabel("Eval step")
plt.ylabel("Loss")
plt.tight_layout()
plt.savefig(os.path.join(cfg.plots_dir, "att_fno_loss.png"))
plt.close()

plt.figure(figsize=(6, 4))
plt.plot(epochs, val_dices, label="Val Dice")
plt.plot(epochs, val_ious, label="Val IoU")
plt.legend()
plt.grid(True)
plt.xlabel("Eval step")
plt.ylabel("Score")
plt.tight_layout()
plt.savefig(os.path.join(cfg.plots_dir, "att_fno_metrics.png"))
plt.close()


In [ ]:
# 10.3 Predikciók vizualizálása és mentése

def visualize_and_save_predictions(model, loader, num_samples=4, save_path=None):
    model.eval()
    images, masks = next(iter(loader))
    images = images.to(device)
    masks = masks.to(device)

    with torch.no_grad():
        outputs = model(images)
        preds = torch.argmax(outputs, dim=1)

    images = images.cpu().numpy()
    masks = masks.cpu().numpy()
    preds = preds.cpu().numpy()

    plt.figure(figsize=(12, num_samples * 3))
    for i in range(num_samples):
        plt.subplot(num_samples, 3, i * 3 + 1)
        plt.imshow(images[i][0], cmap="gray")
        plt.title("Input Image")
        plt.axis("off")

        plt.subplot(num_samples, 3, i * 3 + 2)
        plt.imshow(masks[i], cmap="gray")
        plt.title("Ground Truth")
        plt.axis("off")

        plt.subplot(num_samples, 3, i * 3 + 3)
        plt.imshow(preds[i], cmap="gray")
        plt.title("Model Prediction")
        plt.axis("off")

    plt.tight_layout()
    if save_path is not None:
        plt.savefig(save_path)
        plt.close()
    else:
        plt.show()


best_model = AttentionUNetFNO(in_channels=1, out_channels=cfg.num_classes, modes=4).to(device)
checkpoint_path = os.path.join(cfg.model_dir, "run1_att_fno_best_model.pt")
checkpoint = torch.load(checkpoint_path, map_location=device)
best_model.load_state_dict(checkpoint["model_state_dict"])

preds_path = os.path.join(cfg.preds_dir, "run1_att_fno_preds.png")
visualize_and_save_predictions(best_model, val_loader, num_samples=4, save_path=preds_path)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os

# Feltételezzük, hogy result_att_fno már létezik (train_model_with_logging visszatérési értéke)
# és cfg, cfg.plots_dir is definiálva van, ahogy a notebookban.

train_losses = np.array(result_att_fno["train_history"])[:, 0]
train_dices  = np.array(result_att_fno["train_history"])[:, 1]

val_losses = np.array(result_att_fno["val_history"])[:, 0]
val_dices  = np.array(result_att_fno["val_history"])[:, 1]
val_ious   = np.array(result_att_fno["val_history"])[:, 2]

epochs = np.arange(1, len(train_losses) + 1)

best_val_dice = result_att_fno["best_val_dice"]
# Ha lesz külön teszt Dice/IoU, ide beírhatod:
test_dice = None
test_iou  = None

plt.figure(figsize=(18, 5))

# 1. subplot – Train / Val loss
plt.subplot(1, 3, 1)
plt.plot(epochs, train_losses, label="Train Loss", color="tab:blue")
plt.plot(epochs, val_losses, label="Val Loss", color="tab:orange")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.grid(True, alpha=0.3)
plt.legend()

# 2. subplot – Dice (Train / Val)
plt.subplot(1, 3, 2)
plt.plot(epochs, train_dices, label="Train Dice", color="tab:blue")
plt.plot(epochs, val_dices, label="Val Dice", color="tab:orange")
plt.xlabel("Epoch")
plt.ylabel("Dice score")
plt.title("Dice Score")
plt.grid(True, alpha=0.3)
plt.legend()

# 3. subplot – Val Dice / Val IoU
plt.subplot(1, 3, 3)
plt.plot(epochs, val_dices, label="Val Dice", color="tab:blue")
plt.plot(epochs, val_ious, label="Val IoU", color="tab:orange")
plt.xlabel("Epoch")
plt.ylabel("Score")
plt.title("Validation Metrics")
plt.grid(True, alpha=0.3)
plt.legend()

# Fő cím – hasonló stílusban, mint a példaképen
if test_dice is not None and test_iou is not None:
    suptitle = (
        f"Training History: AttentionUNetFNO "
        f"- Best Val Dice: {best_val_dice:.4f} "
        f"Test Dice: {test_dice:.4f}, Test IoU: {test_iou:.4f}"
    )
else:
    suptitle = (
        f"Training History: AttentionUNetFNO "
        f"- Best Val Dice: {best_val_dice:.4f}"
    )

plt.suptitle(suptitle, fontsize=14, y=1.02)

plt.tight_layout()

save_path = os.path.join(cfg.plots_dir, "att_unet_fno_training_history.png")
plt.savefig(save_path, bbox_inches="tight", dpi=150)
plt.close()

print("Mentve ide:", save_path)


Mentve ide: /content/drive/MyDrive/Brain MRI/AttentionUnet_FNO/plots/att_unet_fno_training_history.png


In [ ]:
import torch
import numpy as np

# --- Legjobb modell betöltése ---
best_model = AttentionUNetFNO(in_channels=1, out_channels=cfg.num_classes, modes=4).to(device)

checkpoint_path = os.path.join(cfg.model_dir, "run1_att_fno_best_model.pt")
checkpoint = torch.load(checkpoint_path, map_location=device)
best_model.load_state_dict(checkpoint["model_state_dict"])

best_model.eval()

# --- Test Dice / IoU számítása ---
all_dice = []
all_iou = []

with torch.no_grad():
    for images, masks in val_loader:   # <-- ha lesz külön test_loader, cseréld le arra
        images = images.to(device)
        masks = masks.to(device)

        outputs = best_model(images)

        # Dice és IoU a PDF-ben definiált függvényekkel
        batch_dice = dice_score(outputs, masks, num_classes=cfg.num_classes)
        batch_iou  = compute_iou(outputs, masks, num_classes=cfg.num_classes)

        all_dice.extend(batch_dice)
        all_iou.extend(batch_iou)

test_dice = float(np.mean(all_dice))
test_iou  = float(np.mean(all_iou))

print("Test Dice:", test_dice)
print("Test IoU :", test_iou)


Test Dice: 0.9259661978716668
Test IoU : 0.8880608493501801


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os

# --- A PDF-ben szereplő adatok ---
train_losses = np.array(result_att_fno["train_history"])[:, 0]
train_dices  = np.array(result_att_fno["train_history"])[:, 1]

val_losses = np.array(result_att_fno["val_history"])[:, 0]
val_dices  = np.array(result_att_fno["val_history"])[:, 1]
val_ious   = np.array(result_att_fno["val_history"])[:, 2]

epochs = np.arange(1, len(train_losses) + 1)

best_val_dice = result_att_fno["best_val_dice"]

# --- A teszt eredményeid (már kiszámolva) ---
test_dice = 0.9259661978716668
test_iou  = 0.8880608493501801

# --- Ábra rajzolása ---
plt.figure(figsize=(18, 5))

# 1. subplot – Train / Val loss
plt.subplot(1, 3, 1)
plt.plot(epochs, train_losses, label="Train Loss", color="tab:blue")
plt.plot(epochs, val_losses, label="Val Loss", color="tab:orange")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.grid(True, alpha=0.3)
plt.legend()

# 2. subplot – Dice (Train / Val)
plt.subplot(1, 3, 2)
plt.plot(epochs, train_dices, label="Train Dice", color="tab:blue")
plt.plot(epochs, val_dices, label="Val Dice", color="tab:orange")
plt.xlabel("Epoch")
plt.ylabel("Dice score")
plt.title("Dice Score")
plt.grid(True, alpha=0.3)
plt.legend()

# 3. subplot – Val Dice / Val IoU
plt.subplot(1, 3, 3)
plt.plot(epochs, val_dices, label="Val Dice", color="tab:blue")
plt.plot(epochs, val_ious, label="Val IoU", color="tab:orange")
plt.xlabel("Epoch")
plt.ylabel("Score")
plt.title("Validation Metrics")
plt.grid(True, alpha=0.3)
plt.legend()

# --- Fő cím: Best Val Dice + Test Dice + Test IoU ---
plt.suptitle(
    f"Training History: Attention U-Net FNO – "
    f"Best Val Dice: {best_val_dice:.4f} | "
    f"Test Dice: {test_dice:.4f} | Test IoU: {test_iou:.4f}",
    fontsize=14,
    y=1.05
)

plt.tight_layout()

save_path = os.path.join(cfg.plots_dir, "att_unet_fno_training_history_full.png")
plt.savefig(save_path, bbox_inches="tight", dpi=150)
plt.close()

print("Plot mentve ide:", save_path)


Plot mentve ide: /content/drive/MyDrive/Brain MRI/AttentionUnet_FNO/plots/att_unet_fno_training_history_full.png
